# 🚀 Image Captioning — Model B: YOLO + BLIP-Large + CLIP + SentenceTransformer Re-ranking
### COCO val2017 — BLEU, ROUGE, CIDEr, Precision/Recall/F1, Hallucination Rate + Visual Samples

In [ ]:
!pip install ultralytics transformers sentence-transformers opencv-python \
    scikit-learn tqdm pycocoevalcap nltk matplotlib seaborn faiss-cpu -q
print('✅ Dependencies installed')

In [ ]:
import os
if not os.path.exists('val2017'):
    !wget -q http://images.cocodataset.org/zips/val2017.zip
    !unzip -q -o val2017.zip
if not os.path.exists('annotations'):
    !wget -q http://images.cocodataset.org/annotations/annotations_trainval2017.zip
    !unzip -q -o annotations_trainval2017.zip
print('✅ Dataset ready')

In [ ]:
import json, time, numpy as np
from tqdm import tqdm
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
from ultralytics import YOLO
from transformers import BlipProcessor, BlipForConditionalGeneration
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

plt.rcParams.update({
    'figure.facecolor': '#0f0f0f',
    'axes.facecolor':   '#1a1a2e',
    'axes.edgecolor':   '#444',
    'axes.labelcolor':  '#e0e0e0',
    'text.color':       '#e0e0e0',
    'xtick.color':      '#aaa',
    'ytick.color':      '#aaa',
    'grid.color':       '#333',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
})
ACCENT  = '#00d2ff'
ACCENT2 = '#ff6b6b'
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Device: {DEVICE}')

In [ ]:
## ── Section 1: Load Annotations ──
ANN_FILE  = 'annotations/captions_val2017.json'
INST_FILE = 'annotations/instances_val2017.json'

with open(ANN_FILE) as f:  cap_data  = json.load(f)
with open(INST_FILE) as f: inst_data = json.load(f)

image_id_to_name = {img['id']: img['file_name'] for img in cap_data['images']}
image_to_captions = {}
for ann in cap_data['annotations']:
    fname = image_id_to_name[ann['image_id']]
    image_to_captions.setdefault(fname, []).append(ann['caption'])

cat_id_to_name = {c['id']: c['name'] for c in inst_data['categories']}
image_to_objects = {}
_img_id_objs = {}
for ann in inst_data['annotations']:
    _img_id_objs.setdefault(ann['image_id'], set()).add(cat_id_to_name[ann['category_id']])
for img in inst_data['images']:
    if img['id'] in _img_id_objs:
        image_to_objects[img['file_name']] = list(_img_id_objs[img['id']])
object_vocab = set(obj for objs in image_to_objects.values() for obj in objs)

print(f'✅ {len(image_to_captions)} images | {sum(len(v) for v in image_to_captions.values())} captions')

In [ ]:
## ── Section 2: Load Models ──
print('Loading YOLOv8n...')
yolo = YOLO('yolov8n.pt')

MODEL_NAME = 'Salesforce/blip-image-captioning-large'
print(f'Loading BLIP ({MODEL_NAME})...')
blip_processor = BlipProcessor.from_pretrained(MODEL_NAME)
blip_model     = BlipForConditionalGeneration.from_pretrained(MODEL_NAME).to(DEVICE)
blip_model.eval()

print('Loading SentenceTransformer...')
sem_model = SentenceTransformer('all-MiniLM-L6-v2')

print('Pre-encoding COCO captions for re-ranking (~30s)...')
all_captions = [cap for caps in image_to_captions.values() for cap in caps]
archive_embeddings = sem_model.encode(
    all_captions, batch_size=256, convert_to_numpy=True, show_progress_bar=True
)
print('✅ All models loaded')

In [ ]:
## ── Section 3: Pipeline Functions ──
CONF_THRESHOLD = 0.30
TOP_K_OBJECTS  = 5
N_CANDIDATES   = 7
ALPHA          = 0.5

def get_yolo_objects(img_path, conf_thresh=CONF_THRESHOLD, top_k=TOP_K_OBJECTS):
    result = yolo(img_path, verbose=False)[0]
    if result.boxes is None or len(result.boxes) == 0:
        return [], result
    detections = [(float(conf), yolo.names[int(cls)])
                  for conf, cls in zip(result.boxes.conf, result.boxes.cls)
                  if float(conf) >= conf_thresh]
    detections.sort(key=lambda x: -x[0])
    seen, unique = set(), []
    for _, label in detections:
        if label not in seen:
            seen.add(label); unique.append(label)
        if len(unique) == top_k: break
    return unique, result

def generate_candidates(img_path, n=N_CANDIDATES):
    image  = Image.open(img_path).convert('RGB')
    inputs = blip_processor(images=image, return_tensors='pt').to(DEVICE)
    caps   = []
    with torch.no_grad():
        ids = blip_model.generate(**inputs, max_new_tokens=60, min_length=8,
                                  num_beams=5, length_penalty=1.2, repetition_penalty=1.3)
        caps.append(blip_processor.decode(ids[0], skip_special_tokens=True))
        for temp in [0.7, 0.8, 0.9, 0.95, 1.0, 1.1][:n-1]:
            ids = blip_model.generate(**inputs, max_new_tokens=60, min_length=8,
                                     do_sample=True, top_p=0.92, temperature=temp,
                                     repetition_penalty=1.3)
            caps.append(blip_processor.decode(ids[0], skip_special_tokens=True))
    return list(dict.fromkeys(caps))

def rerank_captions(candidates, yolo_objects):
    if len(candidates) == 1: return candidates[0]
    emb        = sem_model.encode(candidates, convert_to_numpy=True)
    sem_scores = cosine_similarity(emb, archive_embeddings).mean(axis=1)
    sem_scores = (sem_scores - sem_scores.min()) / (np.ptp(sem_scores) + 1e-9)
    ov_scores  = np.zeros(len(candidates))
    if yolo_objects:
        for i, cap in enumerate(candidates):
            ov_scores[i] = sum(1 for o in yolo_objects if o in cap.lower()) / len(yolo_objects)
    final = ALPHA * ov_scores + (1 - ALPHA) * sem_scores
    return candidates[int(np.argmax(final))]

def caption_image(img_path):
    yolo_objects, yolo_result = get_yolo_objects(img_path)
    candidates = generate_candidates(img_path)
    best       = rerank_captions(candidates, yolo_objects)
    return best, yolo_objects, yolo_result, candidates

def caption_objects_in_text(caption):
    cap_lower = caption.lower()
    return {obj for obj in object_vocab if obj in cap_lower}

print('✅ Pipeline functions defined')

In [ ]:
## ── Section 4: Visual Sample Predictions ──
N_SAMPLES   = 6
sample_imgs = list(image_to_captions.keys())[:N_SAMPLES]

fig = plt.figure(figsize=(20, 14), facecolor='#0d0d1a')
fig.suptitle('Model B — YOLO + BLIP-Large + SentenceTransformer  |  Sample Predictions',
             fontsize=18, color='white', fontweight='bold', y=1.01)

for idx, img_name in enumerate(sample_imgs):
    img_path = os.path.join('val2017', img_name)
    try:
        t0 = time.time()
        best_cap, yolo_objs, yres, all_cands = caption_image(img_path)
        lat = time.time() - t0
        gt_cap = image_to_captions[img_name][0]

        ax = fig.add_subplot(2, 3, idx + 1)
        annotated = yres.plot()
        ax.imshow(annotated[:, :, ::-1])
        ax.axis('off')

        title_text = f'🔮 {best_cap}\n📦 YOLO: {yolo_objs}\n⏱ {lat:.2f}s'
        ax.set_title(title_text, fontsize=8, color='#e0e0e0', pad=6, loc='left', wrap=True)
        ax.set_xlabel(f'GT: {gt_cap[:70]}...', fontsize=7, color='#aaffaa', labelpad=4)
    except Exception as e:
        print(f'Skip {img_name}: {e}')

plt.tight_layout()
plt.savefig('modelB_samples.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print('✅ Visual samples saved to modelB_samples.png')

In [ ]:
## ── Section 5: Precision / Recall / F1 / Hallucination (50 images) ──
TP = FP = FN = 0
times = []
image_list_50 = list(image_to_objects.keys())[:50]

print('Running P/R/F1 on 50 images...')
for img_name in tqdm(image_list_50):
    try:
        img_path = os.path.join('val2017', img_name)
        t0 = time.time()
        cap, yolo_objects, _, _ = caption_image(img_path)
        times.append(time.time() - t0)
        gt   = set(image_to_objects[img_name])
        pred = caption_objects_in_text(cap)
        TP += len(pred & gt)
        FP += len(pred - gt)
        FN += len(gt - pred)
    except Exception as e:
        print(f'  Error: {e}')

precision     = TP / (TP + FP + 1e-9)
recall        = TP / (TP + FN + 1e-9)
f1            = 2 * precision * recall / (precision + recall + 1e-9)
hall_rate     = FP / (TP + FP + 1e-9)
avg_time_prf1 = np.mean(times)

print('\n══════════════════════════════════')
print('  MODEL B — P/R/F1 RESULTS')
print('══════════════════════════════════')
print(f'  Precision          : {precision:.4f}')
print(f'  Recall             : {recall:.4f}')
print(f'  F1                 : {f1:.4f}')
print(f'  Hallucination Rate : {hall_rate:.4f}')
print(f'  Avg Time/image     : {avg_time_prf1:.2f}s')
print('══════════════════════════════════')

In [ ]:
## ── Section 6: BLEU + ROUGE + CIDEr Evaluation (100 images) ──
import nltk
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

from pycocoevalcap.bleu.bleu   import Bleu
from pycocoevalcap.rouge.rouge import Rouge
from pycocoevalcap.cider.cider import Cider

gts, res = {}, {}
latencies = []
image_list_100 = list(image_to_captions.keys())[:100]

print('BLEU/ROUGE/CIDEr evaluation on 100 images...')
for img_name in tqdm(image_list_100):
    try:
        img_path = os.path.join('val2017', img_name)
        t0 = time.time()
        cap, _, _, _ = caption_image(img_path)
        latencies.append(time.time() - t0)
        gts[img_name] = image_to_captions[img_name]
        res[img_name] = [cap]
    except Exception as e:
        print(f'Error: {e}')

bleu_scorer = Bleu(4)
bleu_scores, _ = bleu_scorer.compute_score(gts, res)
rouge = Rouge()
rouge_score, _ = rouge.compute_score(gts, res)
cider = Cider()
cider_score, _ = cider.compute_score(gts, res)

modelB_results = {
    'BLEU-1': bleu_scores[0], 'BLEU-2': bleu_scores[1],
    'BLEU-3': bleu_scores[2], 'BLEU-4': bleu_scores[3],
    'ROUGE-L': rouge_score,  'CIDEr': cider_score,
    'Precision': precision,  'Recall': recall,
    'F1': f1,                'Hallucination_Rate': hall_rate,
    'Avg_Latency': np.mean(latencies),
    'Min_Latency': np.min(latencies),
    'Max_Latency': np.max(latencies),
}

print('\n══════════════════════════════════')
print('  MODEL B — FULL RESULTS')
print('══════════════════════════════════')
for k, v in modelB_results.items():
    print(f'  {k:<22}: {v:.4f}')
print('══════════════════════════════════')

In [ ]:
## ── Section 7: Metric Visualizations ──

fig, axes = plt.subplots(2, 2, figsize=(16, 12), facecolor='#0d0d1a')
fig.suptitle('Model B — YOLO + BLIP-Large + SentenceTransformer  |  Evaluation Metrics',
             fontsize=16, color='white', fontweight='bold')

# ── BLEU bars ──
ax = axes[0, 0]
bleu_names = ['BLEU-1','BLEU-2','BLEU-3','BLEU-4']
bleu_vals  = [modelB_results[k] for k in bleu_names]
colors = ['#00d2ff','#0099cc','#006699','#003366']
bars = ax.bar(bleu_names, bleu_vals, color=colors, edgecolor='#333', width=0.6)
for bar, v in zip(bars, bleu_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{v:.3f}', ha='center', va='bottom', fontsize=10, color='white')
ax.set_ylim(0, max(bleu_vals)*1.35)
ax.set_title('BLEU Scores', color='white', fontsize=13)
ax.set_ylabel('Score', color='white')
ax.grid(axis='y', alpha=0.3)

# ── Precision / Recall / F1 / Hallucination ──
ax2 = axes[0, 1]
prf_names = ['Precision','Recall','F1','Hall.Rate']
prf_vals  = [modelB_results['Precision'], modelB_results['Recall'],
             modelB_results['F1'], modelB_results['Hallucination_Rate']]
prf_colors = ['#00d2ff','#7fff00','#ffd700','#ff6b6b']
bars2 = ax2.bar(prf_names, prf_vals, color=prf_colors, edgecolor='#333', width=0.6)
for bar, v in zip(bars2, prf_vals):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{v:.3f}', ha='center', va='bottom', fontsize=10, color='white')
ax2.set_ylim(0, 1.2)
ax2.set_title('Precision / Recall / F1 / Hallucination Rate', color='white', fontsize=12)
ax2.set_ylabel('Score', color='white')
ax2.grid(axis='y', alpha=0.3)

# ── ROUGE-L + CIDEr gauge ──
ax3 = axes[1, 0]
extra_names  = ['ROUGE-L','CIDEr']
extra_vals   = [modelB_results['ROUGE-L'], modelB_results['CIDEr']]
extra_colors = ['#f7971e','#e040fb']
bars3 = ax3.bar(extra_names, extra_vals, color=extra_colors, edgecolor='#333', width=0.5)
for bar, v in zip(bars3, extra_vals):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
             f'{v:.3f}', ha='center', va='bottom', fontsize=11, color='white', fontweight='bold')
ax3.set_title('ROUGE-L & CIDEr Scores', color='white', fontsize=13)
ax3.set_ylabel('Score', color='white')
ax3.grid(axis='y', alpha=0.3)

# ── Latency distribution ──
ax4 = axes[1, 1]
ax4.hist(latencies, bins=20, color='#00d2ff', edgecolor='#333', alpha=0.85)
ax4.axvline(np.mean(latencies), color='#ff6b6b', linestyle='--',
            linewidth=2, label=f'Mean={np.mean(latencies):.2f}s')
ax4.set_title('Inference Latency Distribution', color='white', fontsize=13)
ax4.set_xlabel('Time (s)', color='white')
ax4.set_ylabel('Count', color='white')
ax4.legend(facecolor='#1a1a2e', labelcolor='white')
ax4.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('modelB_metrics.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print('✅ Metric plots saved to modelB_metrics.png')

In [ ]:
## ── Section 8: Per-Image BLEU Analysis ──
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

per_bleu = []
for k in list(gts.keys())[:50]:
    refs_tok = [r.lower().split() for r in gts[k]]
    hyp_tok  = res[k][0].lower().split()
    sc = sentence_bleu(refs_tok, hyp_tok,
                       smoothing_function=SmoothingFunction().method1)
    per_bleu.append(sc)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor='#0d0d1a')
fig.suptitle('Model B — Per-Image BLEU Analysis (first 50 images)',
             color='white', fontsize=14, fontweight='bold')

axes[0].plot(range(len(per_bleu)), per_bleu, color='#00d2ff', linewidth=1.5, alpha=0.8)
axes[0].fill_between(range(len(per_bleu)), per_bleu, alpha=0.25, color='#00d2ff')
axes[0].axhline(np.mean(per_bleu), color='#ff6b6b', linestyle='--',
                label=f'Mean={np.mean(per_bleu):.3f}')
axes[0].set_title('Per-Image BLEU-1 Trend', color='white', fontsize=12)
axes[0].set_xlabel('Image Index', color='white')
axes[0].set_ylabel('BLEU-1', color='white')
axes[0].legend(facecolor='#1a1a2e', labelcolor='white')
axes[0].grid(alpha=0.3)

axes[1].hist(per_bleu, bins=15, color='#ff6b6b', edgecolor='#333', alpha=0.85)
axes[1].set_title('BLEU-1 Score Distribution', color='white', fontsize=12)
axes[1].set_xlabel('BLEU-1', color='white')
axes[1].set_ylabel('Frequency', color='white')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('modelB_bleu_analysis.png', dpi=150, bbox_inches='tight', facecolor='#0d0d1a')
plt.show()
print(f'✅ Mean per-image BLEU-1: {np.mean(per_bleu):.4f}')

In [ ]:
## ── Save results for comparison ──
import pickle
with open('modelB_results.pkl', 'wb') as f:
    pickle.dump({'metrics': modelB_results, 'per_bleu': per_bleu,
                 'latencies': latencies}, f)
print('✅ Model B results saved to modelB_results.pkl')